# Standardization

In [1]:
import pandas as pd
import re

## Leitura do arquivo

In [2]:
df = pd.read_csv(
    "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/ibge_pnad_continua/tabela5436.csv",
    sep=",",
    encoding="latin1",
    #skiprows=10  # remove cabeçalho do SIDRA
)

In [3]:
print(len(df))
df.head()

27


,ï»¿CÃ³d.,Unidade da FederaÃ§Ã£o,1Âº trimestre 2018,2Âº trimestre 2018,3Âº trimestre 2018,4Âº trimestre 2018,1Âº trimestre 2019,2Âº trimestre 2019,3Âº trimestre 2019,4Âº trimestre 2019,1Âº trimestre 2020,2Âº trimestre 2022,3Âº trimestre 2022,4Âº trimestre 2022
0,11,RondÃ´nia,2699,2663,2787,2719,2601,2622,2739,2709,2662,2658,2755,2900
1,12,Acre,2352,2401,2466,2476,2578,2544,2510,2595,2461,2641,2587,2538
2,13,Amazonas,2442,2439,2321,2277,2335,2288,2352,2294,2320,2258,2333,2409
3,14,Roraima,3185,3059,3011,2976,2935,3136,3146,2926,2995,2708,2602,2936
4,15,ParÃ¡,2109,2063,2096,2052,2082,2027,2093,2156,2113,2125,2217,2243


In [4]:
df = df.rename(columns={
    df.columns[0]: "cod_uf",
    df.columns[1]: "uf"
})

In [5]:
df.columns = df.columns.str.encode('latin1').str.decode('utf-8')
df['uf'] = df['uf'].str.encode('latin1').str.decode('utf-8')

- First validation

In [6]:
df.head()

,cod_uf,uf,1º trimestre 2018,2º trimestre 2018,3º trimestre 2018,4º trimestre 2018,1º trimestre 2019,2º trimestre 2019,3º trimestre 2019,4º trimestre 2019,1º trimestre 2020,2º trimestre 2022,3º trimestre 2022,4º trimestre 2022
0,11,Rondônia,2699,2663,2787,2719,2601,2622,2739,2709,2662,2658,2755,2900
1,12,Acre,2352,2401,2466,2476,2578,2544,2510,2595,2461,2641,2587,2538
2,13,Amazonas,2442,2439,2321,2277,2335,2288,2352,2294,2320,2258,2333,2409
3,14,Roraima,3185,3059,3011,2976,2935,3136,3146,2926,2995,2708,2602,2936
4,15,Pará,2109,2063,2096,2052,2082,2027,2093,2156,2113,2125,2217,2243


In [8]:
anos = sorted(set([
    re.search(r'\d{4}', col).group()
    for col in df.columns if re.search(r'\d{4}', col)
]))

In [9]:
for ano in anos:
    cols_ano = [col for col in df.columns if ano in col]
    df[ano] = df[cols_ano].sum(axis=1)

- Second validation

In [10]:
df.head()

,cod_uf,uf,1º trimestre 2018,2º trimestre 2018,3º trimestre 2018,4º trimestre 2018,1º trimestre 2019,2º trimestre 2019,3º trimestre 2019,4º trimestre 2019,1º trimestre 2020,2º trimestre 2022,3º trimestre 2022,4º trimestre 2022,2018,2019,2020,2022
0,11,Rondônia,2699,2663,2787,2719,2601,2622,2739,2709,2662,2658,2755,2900,10868,10671,2662,8313
1,12,Acre,2352,2401,2466,2476,2578,2544,2510,2595,2461,2641,2587,2538,9695,10227,2461,7766
2,13,Amazonas,2442,2439,2321,2277,2335,2288,2352,2294,2320,2258,2333,2409,9479,9269,2320,7000
3,14,Roraima,3185,3059,3011,2976,2935,3136,3146,2926,2995,2708,2602,2936,12231,12143,2995,8246
4,15,Pará,2109,2063,2096,2052,2082,2027,2093,2156,2113,2125,2217,2243,8320,8358,2113,6585


In [11]:
cols_trimestres = [col for col in df.columns if 'trimestre' in col]
df = df.drop(columns=cols_trimestres)

In [12]:
df_long = df.melt(
    id_vars=['cod_uf', 'uf'],
    var_name='ano',
    value_name='renda_media'
)

- Third validation

In [13]:
print(len(df_long))
df_long.head()

108


,cod_uf,uf,ano,renda_media
0,11,Rondônia,2018,10868
1,12,Acre,2018,9695
2,13,Amazonas,2018,9479
3,14,Roraima,2018,12231
4,15,Pará,2018,8320


### Writing the dataset

In [14]:
df_long.to_parquet('/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/ibge_pnad_continua/ibge_pnad_continua.parquet', index=False)